# Past fire risk from ERA5-Land

This notebook calculates the **past fire-risk layer** for Deliblato sands Special nature reserve in Serbia using:

- static fire-susceptibility raster `StaticV4` produced in ArcGIS Pro Model builder
- ERA5-Land hourly meteorological variables
- ERA5 total cloud cover
- selected date and 13:00 UTC

The output is an interactive map.

In [ ]:
# Install only if needed
# !pip install earthengine-api geemap

In [2]:
import ee
import geemap

In [3]:
# Authenticate and initialize Earth Engine

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    # CHANGE PROJECT NAME
    ee.Initialize(project='ee-minucher')

Enter verification code: 4/1AeoWuM9_7Uj-ggjTZbNpxN3aHWzLHLYThmCXePnRV-6HXvkzmzDsqrPi_Go

Successfully saved authorization token.


In [4]:
# GEE assets

roi = ee.FeatureCollection("projects/ee-minucher/assets/DeliblatoROI_SQFIN")
roisrp = ee.FeatureCollection("projects/ee-minucher/assets/SRPDeliblatoWGS84")
stat4 = ee.Image("projects/ee-minucher/assets/StaticV4")

roi_geom = roi.geometry()
roisrp_geom = roisrp.geometry()

In [5]:
# User settings

DATE = "2024-08-01"
HOUR_UTC = 13

In [6]:
# Datasets

era5_land = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
era5 = ee.ImageCollection("ECMWF/ERA5/HOURLY")

In [7]:
def norm(img, min_val, max_val):
    return (
        img.subtract(min_val)
        .divide(ee.Number(max_val).subtract(min_val))
        .clamp(0, 1)
    )


def mean_over_area(img, geom, scale=11132):
    result = img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geom,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9,
    )
    return result.values().get(0)

In [8]:
# Select ERA5-Land and ERA5 image for selected hour

target = ee.Date(f"{DATE}T{HOUR_UTC:02d}:00:00")
target_end = target.advance(1, "hour")

era_land_img = ee.Image(
    era5_land
    .filterDate(target, target_end)
    .first()
)

era_img = ee.Image(
    era5
    .filterDate(target, target_end)
    .first()
)

print("Selected date:", DATE)
print("Selected hour:", HOUR_UTC, "UTC")
print("ERA5-Land bands:", era_land_img.bandNames().getInfo()[:10], "...")

Selected date: 2024-08-01
Selected hour: 13 UTC
ERA5-Land bands: ['dewpoint_temperature_2m', 'temperature_2m', 'skin_temperature', 'soil_temperature_level_1', 'soil_temperature_level_2', 'soil_temperature_level_3', 'soil_temperature_level_4', 'lake_bottom_temperature', 'lake_ice_depth', 'lake_ice_temperature'] ...


In [9]:
# Raw variables

t_c = era_land_img.select("temperature_2m").subtract(273.15).rename("air_temperature_C")
td_c = era_land_img.select("dewpoint_temperature_2m").subtract(273.15).rename("dewpoint_temperature_C")

solar_w = (
    era_land_img
    .select("surface_solar_radiation_downwards_hourly")
    .divide(3600)
    .rename("solar_radiation_Wm2")
)

rain_mm = (
    era_land_img
    .select("total_precipitation_hourly")
    .multiply(1000)
    .rename("precipitation_mm")
)

u10 = era_land_img.select("u_component_of_wind_10m")
v10 = era_land_img.select("v_component_of_wind_10m")
wind = u10.pow(2).add(v10.pow(2)).sqrt().rename("wind_speed_ms")

soil_moisture = era_land_img.select("volumetric_soil_water_layer_1").rename("soil_moisture_layer1")
soil_temp_c = era_land_img.select("soil_temperature_level_1").subtract(273.15).rename("soil_temperature_C")

cloud = era_img.select("total_cloud_cover").rename("cloud_cover")

In [10]:
# Derived moisture variables: relative humidity and VPD

es = t_c.expression(
    "0.6108 * exp(17.27 * T / (T + 237.3))",
    {"T": t_c}
).rename("saturation_vapor_pressure_kPa")

ea = td_c.expression(
    "0.6108 * exp(17.27 * Td / (Td + 237.3))",
    {"Td": td_c}
).rename("actual_vapor_pressure_kPa")

rh = ea.divide(es).multiply(100).clamp(0, 100).rename("relative_humidity_percent")
vpd = es.subtract(ea).max(0).rename("vapor_pressure_deficit_kPa")

In [11]:
# Normalized risk terms

t_risk = norm(t_c, 15, 38)
vpd_risk = norm(vpd, 0.5, 3.5)
wind_risk = norm(wind, 1, 10)
solar_risk = norm(solar_w, 150, 850)

rain_dry_risk = ee.Image(1).subtract(norm(rain_mm, 0.2, 2.0))
soil_dry_risk = ee.Image(1).subtract(norm(soil_moisture, 0.12, 0.38))
soil_temp_risk = norm(soil_temp_c, 10, 32)
cloud_clear_risk = ee.Image(1).subtract(norm(cloud, 0.3, 1.0))

In [12]:
# Weather index and dynamic modifier

weather_index = (
    t_risk.multiply(0.20)
    .add(vpd_risk.multiply(0.20))
    .add(wind_risk.multiply(0.20))
    .add(solar_risk.multiply(0.10))
    .add(rain_dry_risk.multiply(0.10))
    .add(soil_dry_risk.multiply(0.10))
    .add(soil_temp_risk.multiply(0.05))
    .add(cloud_clear_risk.multiply(0.05))
    .rename("weather_index")
)

modifier = ee.Image(0.2).add(weather_index.multiply(1.6)).rename("dynamic_modifier")

# Wet-condition suppression
modifier = modifier.where(rain_mm.gt(2.0), 0.15)
modifier = modifier.where(rh.gt(95), 0.25)
modifier = modifier.where(rain_mm.gt(0.5).And(rh.gt(85)), 0.35)

In [13]:
# Final fire risk

fire_risk = (
    stat4
    .multiply(modifier)
    .clamp(0, 100)
    .rename("past_fire_risk")
)

fire_risk = fire_risk.updateMask(fire_risk.gt(0))

In [14]:
# Print mean diagnostic values over the protected area

diagnostics = {
    "Air temperature (°C)": t_c,
    "Dewpoint temperature (°C)": td_c,
    "Relative humidity (%)": rh,
    "VPD (kPa)": vpd,
    "Wind speed (m/s)": wind,
    "Solar radiation (W/m²)": solar_w,
    "Precipitation (mm)": rain_mm,
    "Soil moisture": soil_moisture,
    "Soil temperature (°C)": soil_temp_c,
    "Cloud cover": cloud,
    "Weather index": weather_index,
    "Dynamic modifier": modifier,
    "Final fire risk": fire_risk,
}

for name, img in diagnostics.items():
    value = mean_over_area(img, roisrp_geom).getInfo()
    print(f"{name}: {value:.2f}" if value is not None else f"{name}: n/a")

Air temperature (°C): 33.11
Dewpoint temperature (°C): 8.99
Relative humidity (%): 22.71
VPD (kPa): 3.91
Wind speed (m/s): 1.94
Solar radiation (W/m²): 793.84
Precipitation (mm): 0.00
Soil moisture: 0.17
Soil temperature (°C): 33.74
Cloud cover: 0.02
Weather index: 0.75
Dynamic modifier: 1.40
Final fire risk: 53.66


In [15]:
# Interactive map

risk_palette = [
    "#006400", "#228B22", "#7FBF3F",
    "#ADFF2F", "#FFFF00", "#FFD700",
    "#FFA500", "#FF7F00", "#FF4500",
    "#8B0000",
]

m = geemap.Map()
m.centerObject(roisrp, 10)
m.add_basemap("HYBRID")

m.addLayer(
    fire_risk.clip(roi_geom),
    {"min": 0, "max": 100, "palette": risk_palette},
    f"Past fire risk {DATE} {HOUR_UTC}:00 UTC",
)

m.addLayer(
    ee.Image().paint(roisrp, 1, 2),
    {"palette": ["00FFFF"]},
    "Protected area boundary",
)

m.addLayer(
    weather_index.clip(roi_geom),
    {"min": 0, "max": 1, "palette": ["blue", "cyan", "yellow", "orange", "red"]},
    "Weather index",
    shown=False,
)

m.addLayer(
    modifier.clip(roi_geom),
    {"min": 0.15, "max": 1.8, "palette": ["blue", "cyan", "yellow", "orange", "red"]},
    "Dynamic modifier",
    shown=False,
)

m.addLayerControl()
m

Map(center=[44.89675093876225, 21.122467464313274], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# Optional export to GeoTIFF
# Uncomment and run if needed.

# geemap.ee_export_image(
#     fire_risk.clip(roi_geom),
#     filename=f"past_fire_risk_{DATE}_{HOUR_UTC:02d}UTC.tif",
#     scale=100,
#     region=roi_geom,
#     file_per_band=False,
# )